In [ ]:
import time
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from numpy.polynomial.legendre import leggauss

                
jax.config.update("jax_enable_x64", True)

# ==========================================
                        
# ==========================================
sigma_inf = 1/300.0      
eta = 0.3               
tau_cc = 0.05            
C = 0.8         

freqs_np = np.logspace(-2, 5, 100)
omega = 2.0 * np.pi * freqs_np
s_vals = jnp.array(1j * omega, dtype=jnp.complex128)

                                 
F_true = 1.0 / (1.0 + (s_vals * tau_cc)**C)
sigma_true = sigma_inf * (1.0 - eta * F_true)


In [ ]:

# ==========================================
          
# ==========================================

                          
def fit_trf(s_vals, F_target, n_poles=8):
    omega_np = np.abs(s_vals.imag)
    s_np = 1j * omega_np
    target_np = np.asarray(F_target)
    
    def residual(params):
        tau_k = params[:n_poles]
        w_k = params[n_poles:-1]
        d0 = params[-1]
        # F_approx = d0 + sum( w_k / (1 + s*tau_k) )
        pred = d0 + np.sum(w_k[None, :] / (1.0 + s_np[:, None] * tau_k[None, :]), axis=1)
        err = np.concatenate([np.real(pred - target_np), np.imag(pred - target_np)])
        weight = 1.0 / (np.abs(target_np) + 1e-5)
        return err * np.concatenate([weight, weight])

    p0 = np.concatenate([np.logspace(-3, 1, n_poles), np.ones(n_poles)/n_poles, [0.0]])
    res = least_squares(residual, p0, bounds=(0, np.inf), method='trf', ftol=1e-8)
    
    d0 = res.x[-1]
    tau_k = res.x[:n_poles]
    w_k = res.x[n_poles:-1]
                                               
    p_k = -1.0 / tau_k
    r_k = w_k / tau_k
    return d0, p_k, r_k

                                     
def fit_gauss_legendre(tau_cc, c_cc, n_poles=8):
    x, w = leggauss(n_poles)
                             
    ln_tau_min, ln_tau_max = np.log(tau_cc) - 13.8, np.log(tau_cc) + 13.8
    y_k = 0.5 * (ln_tau_max - ln_tau_min) * x + 0.5 * (ln_tau_max + ln_tau_min)
    tau_k = np.exp(y_k)
    
                      
    term1 = np.cosh(c_cc * np.log(tau_k / tau_cc))
    term2 = np.cos(c_cc * np.pi)
    G_tau = (1.0 / (2.0 * np.pi * tau_k)) * (np.sin(c_cc * np.pi) / (term1 + term2))
    
    w_k = G_tau * tau_k * w * (0.5 * (ln_tau_max - ln_tau_min))
          
    p_k = -1.0 / tau_k
    r_k = w_k / tau_k
    return 0.0, p_k, r_k

                        
                         
from diffaaable import aaa 

def fit_aaa_method(s_vals, F_target, n_poles=8):
    z_j, f_j, w_j, z_n = aaa(s_vals, F_target, tol=1e-9, mmax=n_poles)
    
    def compute_residues(poles, nodes, values, weights):
        diffs = poles[:, None] - nodes[None, :]
        num = jnp.sum((weights * values)[None, :] / diffs, axis=1)
        den = -jnp.sum(weights[None, :] / (diffs ** 2), axis=1)
        return num / den

    res_all = compute_residues(z_n, z_j, f_j, w_j)
    valid = jnp.real(z_n) < 0
    d0 = jnp.sum(w_j * f_j) / jnp.sum(w_j)
    return d0, z_n[valid], res_all[valid]
    
# ==========================================
                                    
# ==========================================
def decompose_cdrt_gauss(tau_cc, c_cc, n_poles=16, decades=12):
    """
    使用高斯-勒让德求积对连续弛豫谱进行离散化
    decades: 积分覆盖的时间常数范围（以 tau_cc 为中心向两侧延伸）
    """
                  
    nodes, weights = leggauss(n_poles)
    
                                           
    ln_tau_min = np.log(tau_cc) - (decades / 2.0) * np.log(10)
    ln_tau_max = np.log(tau_cc) + (decades / 2.0) * np.log(10)
    
                  
    ln_tau_k = 0.5 * (ln_tau_max - ln_tau_min) * nodes + 0.5 * (ln_tau_max + ln_tau_min)
    tau_k = np.exp(ln_tau_k)
    
                         
    # G(tau) = 1/(2*pi*tau) * sin(c*pi) / (cosh(c*ln(tau/tau_cc)) + cos(c*pi))
    log_ratio = np.log(tau_k / tau_cc)
    term1 = np.cosh(c_cc * log_ratio)
    term2 = np.cos(c_cc * np.pi)
    G_tau = (1.0 / (2.0 * np.pi * tau_k)) * (np.sin(c_cc * np.pi) / (term1 + term2))
    
                         
                                          
    w_mapped = weights * 0.5 * (ln_tau_max - ln_tau_min)
    
                       
    p_k = -1.0 / tau_k
                                                                    
                                                                      
    r_k = G_tau * w_mapped
    
    return 0.0, jnp.array(p_k), jnp.array(r_k)

In [ ]:

# ==========================================
           
# ==========================================
methods = ["TRF-Fit", "Gauss-Legendre", "AAA-Rational"]
results = {}

                
t0 = time.time()
d0_aaa, p_aaa, r_aaa = fit_aaa_method(s_vals, F_true, n_poles=8)
results["AAA-Rational"] = (d0_aaa, p_aaa, r_aaa, time.time()-t0)

                
t0 = time.time()
d0_trf, p_trf, r_trf = fit_trf(s_vals, F_true, n_poles=20)
results["TRF-Fit"] = (d0_trf, p_trf, r_trf, time.time()-t0)

                           
t0 = time.time()
d0_gl, p_gl, r_gl = fit_gauss_legendre(tau_cc, C, n_poles=10)             
results["Gauss-Legendre"] = (d0_gl, p_gl, r_gl, time.time()-t0)

                              
t0 = time.time()
d0_cdrt, p_cdrt, r_cdrt = decompose_cdrt_gauss(tau_cc, C, n_poles=16, decades=14)
results["CDRT"] = (d0_cdrt, p_cdrt, r_cdrt, time.time()-t0)


In [ ]:

# ==========================================
             
# ==========================================
def eval_model(s, d0, p, r):
    return d0 + jnp.sum(r[None, :] / (s[:, None] - p[None, :]), axis=1)

plt.figure(figsize=(20, 6))
styles = {'TRF-Fit': 'r--', 'Gauss-Legendre': 'g-.', 'AAA-Rational': 'b:', 'CDRT': 'm-'}

for name in methods + ["CDRT"]:
    print(name)
    d0, p, r, t = results[name]
    F_approx = eval_model(s_vals, d0, p, r)
    sig_approx = sigma_inf * (1.0 - eta * F_approx)
    
    err = np.abs(sigma_true - sig_approx) / np.abs(sigma_true) * 100
    
          
    plt.subplot(1, 3, 1)
    plt.semilogx(freqs_np, sig_approx.real, styles[name], label=f"{name} (RT: {t:.3f}s)")

          
    plt.subplot(1, 3, 2)
    plt.semilogx(freqs_np, sig_approx.imag, styles[name], label=f"{name}")

          
    plt.subplot(1, 3, 3)
    plt.loglog(freqs_np, err, styles[name][0], label=f"{name} MaxErr: {np.max(err):.2e}%")

          
plt.subplot(1, 3, 1)
plt.semilogx(freqs_np, sigma_true.real, 'k-', lw=2, alpha=0.5, label="True Cole-Cole")
plt.title("Real Part of Conductivity Comparison")
plt.legend(); plt.grid(True, alpha=0.3)

          
plt.subplot(1, 3, 2)
plt.semilogx(freqs_np, sigma_true.imag, 'k-', lw=2, alpha=0.5, label="True Cole-Cole")
plt.title("Imaginary Part of Conductivity Comparison")
plt.legend(); plt.grid(True, alpha=0.3)

     
plt.subplot(1, 3, 3)
plt.title("Relative Error (%) Comparison")
plt.axhline(1e-2, color='k', ls='--', label="0.01% Line")
plt.legend(); plt.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()